In [ ]:
import sys
from pathlib import Path
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

In [ ]:
root_path = Path("/path/to/BrainWear_Kareem")
project_root = root_path / "FYP"
print("Project root:", project_root)

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from datasets.brats2020_png import BraTS2020PNGDataset
from slot_attention.training_2d.slot_attention_2d import SlotClassifier2D

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
# Keys are checkpoint folder names under training_2d/models/checkpoints/
# Values are the display labels shown in the figure.
CHECKPOINT_LABELS = {
    "brats_png_v14a_0.15_test": "Fully supervised",
    "brats_png_v17_weak_0.15_new_norm":   "Weakly supervised",
    "brats_png_v19a_entropy":   "Weakly supervised\n(spatial dice matching)",
}

N_PATIENTS  = 3      # number of random patients to display
SAMPLE_SEED = None   # None = new random patients each run; set an int for reproducibility
SEED = 42            # deterministic slot initialisation

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

ckpt_base = project_root / "slot_attention" / "training_2d" / "models" / "checkpoints"

models = {}   # display_label -> model
hps = {}      # display_label -> hyperparameters

for ckpt_name, label in CHECKPOINT_LABELS.items():
    ckpt_path = ckpt_base / ckpt_name / "ckpt.pt"
    checkpoint = torch.load(ckpt_path, map_location=device)
    hp = checkpoint['hyperparameters']

    model = SlotClassifier2D(
        in_shape=(hp['in_channels'], hp['input_h'], hp['input_w']),
        width=hp['width'],
        num_slots=hp['num_slots'],
        slot_dim=hp['slot_dim'],
        routing_iters=hp['routing_iters'],
        temperature=hp['temp'],
        encoder_depth=hp.get('encoder_depth', 4),
        enc3_init_skip=hp.get('enc3_init_skip', False),
        use_mask_pool_classifier=hp.get('use_mask_pool_classifier', False),
    )
    model.load_state_dict(checkpoint['model_state_dict'])
    model.to(device)
    model.eval()
    model.set_deterministic_slot_init(seed=SEED)

    models[label] = model
    hps[label] = hp
    print(f"  '{label}'  ({ckpt_name})  slots={hp['num_slots']}, slot_dim={hp['slot_dim']}, routing={hp['routing_iters']}")

In [ ]:
import random
from PIL import Image as PILImage

data_dir = root_path / "Processed_BraTS2020_TrainingData_PNG"
print("Data directory:", data_dir)

dataset = BraTS2020PNGDataset(data_dir=str(data_dir), shuffle=False)

# Find all slices that contain all 3 tumour classes (raw BraTS labels 1=NCR/NET, 2=ED, 4=ET)
valid_indices = []
for idx, (_, seg_path) in enumerate(dataset.samples):
    seg_arr = np.array(PILImage.open(seg_path), dtype=np.uint8)
    if {1, 2, 4}.issubset(set(np.unique(seg_arr).tolist())):
        valid_indices.append(idx)

print(f"Found {len(valid_indices)} slices with all 3 tumour classes")

rng = random.Random(SAMPLE_SEED)
chosen_indices = rng.sample(valid_indices, N_PATIENTS)
print(f"Selected indices: {chosen_indices}  (SAMPLE_SEED={SAMPLE_SEED})")

# samples[i] = (test_mri tensor (1,1,H,W), ground_truth_mask array (H,W))
samples = []
for idx in chosen_indices:
    t2, seg = dataset[idx]
    samples.append((t2.unsqueeze(0).to(device), seg.numpy()))

print(f"Loaded {N_PATIENTS} samples, GT labels per sample:")
for i, (_, gt) in enumerate(samples):
    print(f"  [{i}] {np.unique(gt)}")

In [ ]:
# all_results[i] = dict: display_label -> masks tensor (1, num_slots, 1, H, W)
all_results = []

for i, (test_mri, _) in enumerate(samples):
    patient_results = {}
    for label, model in models.items():
        with torch.no_grad():
            recon_combined, recons, masks, slots, mlp_outputs = model(test_mri)
        patient_results[label] = masks
    all_results.append(patient_results)
    print(f"Patient {i}: masks {tuple(masks.shape)}")

In [ ]:
brats_cmap = mcolors.ListedColormap(['black', 'red', 'yellow', 'blue'])
bounds = [-0.5, 0.5, 1.5, 2.5, 3.5]
norm = mcolors.BoundaryNorm(bounds, brats_cmap.N)

n_models = len(all_results[0])
num_slots = next(iter(all_results[0].values())).shape[1]
cols = num_slots + 1   # col 0 = original+GT, cols 1..K = slot masks

image_savedir = project_root / "slot_attention" / "training_2d" / "visualisations"
label_str = "_vs_".join(CHECKPOINT_LABELS.keys())

for patient_num, ((test_mri, ground_truth_mask), results, chosen_idx) in enumerate(
        zip(samples, all_results, chosen_indices)):

    original_img = test_mri[0, 0].cpu().numpy()

    fig, axes = plt.subplots(n_models, cols, figsize=(3 * cols, 3 * n_models))
    if n_models == 1:
        axes = axes[np.newaxis, :]

    for row, (label, masks) in enumerate(results.items()):
        slot_masks = masks[0, :, 0].cpu().numpy()   # (num_slots, H, W)

        # Column 0: original + GT overlay
        axes[row, 0].imshow(original_img, cmap='gray')
        gt_overlay = np.ma.masked_where(ground_truth_mask == 0, ground_truth_mask)
        axes[row, 0].imshow(gt_overlay, cmap=brats_cmap, norm=norm, alpha=0.45, interpolation='none')
        axes[row, 0].axis('off')
        if row == 0:
            axes[row, 0].set_title("Original + GT", fontsize=14)

        # Columns 1+: slot masks
        for i in range(slot_masks.shape[0]):
            axes[row, i + 1].imshow(slot_masks[i], cmap='viridis', vmin=0, vmax=1)
            axes[row, i + 1].axis('off')
            if row == 0:
                axes[row, i + 1].set_title(f"Slot {i + 1}", fontsize=14)

    for row, label in enumerate(results.keys()):
        y = 1 - (row + 0.5) / n_models
        fig.text(0.01, y, label, va='center', ha='left',
                 fontsize=14, fontweight='bold', multialignment='left')

    plt.tight_layout(rect=[0.17, 0, 1, 1])

    save_path = image_savedir / f"comparison_{label_str}_sample{chosen_idx}.png"
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    print(f"Patient {patient_num} (idx {chosen_idx}) saved to {save_path}")
    plt.show()

In [ ]:
from datasets.brainwear_png import BrainWearPNGDataset

bw_data_dir = root_path / "Processed_Brainwear_PNG_fixed_norm"
print("BrainWear data directory:", bw_data_dir)

# score_file=None: no labels needed, just T2 images
bw_dataset = BrainWearPNGDataset(root_dir=str(bw_data_dir))
print(f"BrainWear patients: {len(bw_dataset)}")

# Pick N_PATIENTS random patients
rng_bw = random.Random(SAMPLE_SEED)
bw_chosen_indices = rng_bw.sample(range(len(bw_dataset)), N_PATIENTS)
print(f"Selected indices: {bw_chosen_indices}  (SAMPLE_SEED={SAMPLE_SEED})")

# Use q50 slice (index 2 of the 5-slice stack) as the representative axial slice
bw_samples = []
for idx in bw_chosen_indices:
    t2_stack, _ = bw_dataset[idx]                                    # (5, H, W)
    t2_slice = t2_stack[2].unsqueeze(0).unsqueeze(0).to(device)      # q50 → (1, 1, H, W)
    bw_samples.append(t2_slice)

# ── Inference ──────────────────────────────────────────────────────────────────
bw_all_results = []
for i, test_mri in enumerate(bw_samples):
    patient_results = {}
    for label, model in models.items():
        with torch.no_grad():
            _, _, masks, _, _ = model(test_mri)
        patient_results[label] = masks
    bw_all_results.append(patient_results)
    print(f"Patient {i}: masks {tuple(masks.shape)}")

# ── Plot (one figure per patient) ──────────────────────────────────────────────
n_models  = len(bw_all_results[0])
num_slots = next(iter(bw_all_results[0].values())).shape[1]
cols      = num_slots + 1   # col 0 = original, cols 1..K = slot masks

image_savedir = project_root / "slot_attention" / "training_2d" / "visualisations"
label_str = "_vs_".join(CHECKPOINT_LABELS.keys())

for patient_num, (test_mri, results, chosen_idx) in enumerate(
        zip(bw_samples, bw_all_results, bw_chosen_indices)):

    original_img = test_mri[0, 0].cpu().numpy()

    fig, axes = plt.subplots(n_models, cols, figsize=(3 * cols, 3 * n_models))
    if n_models == 1:
        axes = axes[np.newaxis, :]

    for row, (label, masks) in enumerate(results.items()):
        slot_masks = masks[0, :, 0].cpu().numpy()   # (num_slots, H, W)

        # Column 0: original (no GT segmentation for BrainWear)
        axes[row, 0].imshow(original_img, cmap='gray')
        axes[row, 0].axis('off')
        if row == 0:
            axes[row, 0].set_title("Original", fontsize=14)

        # Columns 1+: slot masks
        for i in range(slot_masks.shape[0]):
            axes[row, i + 1].imshow(slot_masks[i], cmap='viridis', vmin=0, vmax=1)
            axes[row, i + 1].axis('off')
            if row == 0:
                axes[row, i + 1].set_title(f"Slot {i + 1}", fontsize=14)

    for row, label in enumerate(results.keys()):
        y = 1 - (row + 0.5) / n_models
        fig.text(0.01, y, label, va='center', ha='left',
                 fontsize=14, fontweight='bold', multialignment='left')

    plt.tight_layout(rect=[0.17, 0, 1, 1])

    save_path = image_savedir / f"bw_comparison_{label_str}_sample{chosen_idx}.png"
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    print(f"Patient {patient_num} (idx {chosen_idx}) saved to {save_path}")
    plt.show()